In [16]:
pip install PyPDF2


In [17]:
pip install deep_translator

In [18]:
from PyPDF2 import PdfReader
from deep_translator import GoogleTranslator
import re
import pandas as pd
from PyPDF2 import PdfReader
import zipfile
import os

#Ekstrak zip untuk pdf beberapa cv

In [19]:

def extract_zip(zip_path, extract_to="cv_folder"):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("✅ ZIP berhasil diekstrak")


def get_pdf_files(folder):
    pdf_files = []
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".pdf"):
                pdf_files.append(os.path.join(root, file))
    return pdf_files


#PDF Text

In [20]:
def pdf_to_text(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text.lower()

#BASIC INFO

In [21]:
def extract_email(text):
    match = re.findall(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', text)
    return match[0] if match else None


def extract_phone(text):
    match = re.findall(r'(\+62\d{9,13}|08\d{8,12})', text)
    return match[0] if match else None


def extract_name(text):
    lines = text.strip().split("\n")
    return lines[0].strip().title() if lines else None

#SECTION EXTRACTOR

In [22]:
def extract_section(text, section_names, stop_keywords):

    pattern = (
        r'(' + '|'.join(section_names) + r')'
        r'(.*?)'
        r'(?=' + '|'.join(stop_keywords) + r'|$)'
    )

    match = re.search(
        pattern,
        text,
        re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(2).strip()

    return ""

#SKILLS

In [23]:
def extract_summary(text):

    summary = extract_section(
        text,

        section_names=[
            'ringkasan',
            'summary',
            'profil',
            'profile',
            'tentang saya',
            'about me',
            'professional summary',
            'career objective',
            'objective',
            'deskripsi diri'
        ],

        stop_keywords=[
            'pendidikan',
            'education',
            'skills',
            'keahlian',
            'organisasi',
            'experience',
            'work experience',
            'pengalaman kerja',
            'projects',
            'sertifikat',
            'certification'
        ]
    )

    # bersihkan newline
    summary = re.sub(r'\n+', ' ', summary)

    # bersihkan spasi berlebih
    summary = re.sub(r'\s+', ' ', summary)

    return summary.strip()

In [24]:
def extract_skills(text):

    lines = text.split("\n")

    skills = []
    capture = False

    start_keywords = [
        "keahlian",
        "skills",
        "skill",
        "kompetensi",
        "tools",
        "teknologi"
    ]

    stop_keywords = [
        "penghargaan",
        "bahasa",
        "hobi",
        "referensi",
        "pendidikan",
        "pengalaman",
        "organisasi"
    ]

    for line in lines:

        line = line.strip().lower()

        # mulai capture
        if any(k in line for k in start_keywords):
            capture = True
            continue

        # stop capture
        if capture and any(k in line for k in stop_keywords):
            break

        if capture:

            if not line:
                continue

            # skip line terlalu panjang
            if len(line.split()) > 10:
                continue

            # bersihkan bullet
            line = re.sub(r'^[•●▪\-]+', '', line).strip()

            skills.append(line)

    # fallback
    if not skills:

        for line in lines:

            line = line.strip().lower()

            if not line:
                continue

            # kandidat skill pendek
            if 1 <= len(line.split()) <= 5:

                if any(k in line for k in [
                    "universitas",
                    "pendidikan",
                    "pengalaman kerja",
                    "organization",
                    "address",
                    "email"
                ]):
                    continue

                skills.append(line)

    # remove duplicate
    skills = list(set(skills))

    return skills

#PENDIDIKAN

In [25]:
def extract_education(text):

    edu_text = extract_section(
        text,

        section_names=[
            'pendidikan',
            'education',
            'academic background',
            'riwayat pendidikan'
        ],

        stop_keywords=[
            'pengalaman kerja',
            'experience',
            'skills',
            'keahlian',
            'organisasi',
            'projects',
            'sertifikat',
            'certification'
        ]
    )

    edu_text = re.sub(r'\n+', ' ', edu_text)
    edu_text = re.sub(r'\s+', ' ', edu_text)

    # degree detection
    degree = None

    if any(k in edu_text.lower() for k in ['sarjana', 's1', 'bachelor']):
        degree = 'S1'

    elif any(k in edu_text.lower() for k in ['s2', 'master']):
        degree = 'S2'

    elif any(k in edu_text.lower() for k in ['d3', 'diploma']):
        degree = 'D3'

    elif 'sma' in edu_text.lower():
        degree = 'SMA'

    # university detection
    university_patterns = [
        r'universitas[\w\s]+',
        r'institut[\w\s]+',
        r'politeknik[\w\s]+',
        r'university[\w\s]+'
    ]

    university = None

    for pattern in university_patterns:

        match = re.search(pattern, edu_text, re.IGNORECASE)

        if match:
            university = match.group().strip()
            break

    return {
        "degree": degree,
        "university": university,
        "education_text": edu_text
    }

#PENGALAMAN

In [26]:
def extract_experience(text):

    exp_text = extract_section(
        text,
        section_names=[
            'pengalaman kerja',
            'work experience',
            'experience',
            'professional experience',
            'internship',
            'pengalaman'
        ],
        stop_keywords=[
            'pendidikan',
            'education',
            'skills',
            'keahlian',
            'organisasi',
            'projects',
            'sertifikat',
            'certification'
        ]
    )

    # bersihin newline berlebih
    exp_text = re.sub(r'\n+', ' ', exp_text)

    return exp_text.strip()

#ROLE

In [27]:
def extract_role(text):

    roles = [
        "data analyst",
        "data scientist",
        "software engineer",
        "frontend developer",
        "backend developer",
        "teacher",
        "admin",
        "accountant",
        "designer",
        "marketing"
    ]

    text = text.lower()

    found_roles = []

    for r in roles:
        if r in text:
            found_roles.append(r)

    return found_roles

#MAIN PARSER

In [35]:
def parse_cv(file_path):

    # extract raw text
    text = pdf_to_text(file_path)

    # basic info
    name = extract_name(text)
    email = extract_email(text)
    phone = extract_phone(text)

    # extract sections
    skills = extract_skills(text)

    summary = extract_summary(text)

    experience = extract_experience(text)

    education = extract_education(text)

    # gabungkan skills
    skills_text = ", ".join(skills)

    # full profile
    candidate_profile = f"""
    Summary:
    {summary}

    Experience:
    {experience}

    Skills:
    {skills_text}

    Education:
    {education["education_text"]}
    """

    # clean whitespace
    candidate_profile = re.sub(r'\s+', ' ', candidate_profile).strip()

    # final data
    data = {

        "candidate_name": name if name else "lorem",

        "email": email if email else "lorem",

        "phone": phone if phone else "lorem",

        "skills": skills_text if skills_text else "lorem",

        # KHUSUS SUMMARY
        "summary": summary if summary else "NONE",

        "experience": experience if experience else "lorem",

        "degree": (
            education["degree"]
            if education["degree"]
            else "lorem"
        ),

        "university": (
            education["university"]
            if education["university"]
            else "lorem"
        ),

        "education": (
            education["education_text"]
            if education["education_text"]
            else "lorem"
        ),

        "text": candidate_profile if candidate_profile else "lorem"
    }

    return data

#Data to CSV

In [36]:
def save_to_csv(data, output="hasil dari cv.csv"):
    df = pd.DataFrame([data])


    file_exists = os.path.isfile(output)

    df.to_csv(
        output,
        mode='a',
        header=not file_exists,
        index=False
    )


In [37]:
if __name__ == "__main__":

    zip_path = "/content/cv pdf.zip"

    extract_zip(zip_path)

    pdf_files = get_pdf_files("cv_folder")

    print("Jumlah CV:", len(pdf_files))

    all_results = []

    for i, pdf in enumerate(pdf_files):

        try:

          print(f"[{i+1}/{len(pdf_files)}] Processing: {pdf}")

          result = parse_cv(pdf)

          print(result)  # DEBUG

          all_results.append(result)

        except Exception as e:

          print(f"ERROR parsing {pdf}: {e}")

    # save sekali di akhir
    df = pd.DataFrame(all_results)

    df.to_csv("parsed_cv.csv", index=False)

    print("Selesai parsing semua CV")

✅ ZIP berhasil diekstrak
Jumlah CV: 366
[1/366] Processing: cv_folder/CV Siti Nurhaliza by Naevaweb (5).pdf
{'candidate_name': 'Siti Nurhaliza', 'email': 'siti.nurhaliza@example.com', 'phone': '+6281234567890', 'skills': 'penanganan keluhan, komunikasi efektif, penyelesaian masalah, manajemen panggilan, empati, zendesk, microsoft office, kepuasan pelanggan, manajemen waktu, freshdesk', 'summary': 'NONE', 'experience': 'lebih dari 2 tahun dalam menangani pertanyaan dan  keluhan pelanggan secara efisien. terbiasa bekerja di bawah tekanan dengan sikap ramah dan solusi yang  cepat. memiliki keterampilan komunikasi yang baik dan mampu membangun hubungan positif dengan  pelanggan. berkomitmen untuk memberikan pelayanan terbaik dan menjaga kepuasan pelanggan.', 'degree': 'SMA', 'university': 'lorem', 'education': 'sma negeri 2 jakarta agustus 2018 - juli 2021 sma negeri 2 jakarta jakarta jurusan: ips nilai rata-rata: 85/100', 'text': 'Summary: Experience: lebih dari 2 tahun dalam menangani pe

In [38]:
cv_compress = pd.read_csv("/content/parsed_cv.csv")
cv_compress.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,education,text
0,Siti Nurhaliza,siti.nurhaliza@example.com,6281234567890,"penanganan keluhan, komunikasi efektif, penyel...",NONE,lebih dari 2 tahun dalam menangani pertanyaan ...,SMA,lorem,sma negeri 2 jakarta agustus 2018 - juli 2021 ...,Summary: Experience: lebih dari 2 tahun dalam ...
1,Rina Aulia,rina.aulia@example.com,6281234567890,"komunikasi efektif, crm software, salesforce, ...",NONE,3 tahun dalam memberikan pelayanan yang optima...,D3,universitas lorem ipsum jakarta lulus dengan p...,d3 administrasi bisnis september 2018 - juli 2...,Summary: Experience: 3 tahun dalam memberikan ...
2,Anna Suryani,anna.suryani@example.com,6281234567890,"problem-solving, intern midwife august 2023 - ...",NONE,d in providing comprehensive care to expectant...,S1,university of indonesia jakarta focused on mat...,", and teamwork within a healthcare environment...",Summary: Experience: d in providing comprehens...
3,Andi Pratama,andi.pratama@example.com,6281123456789,"sistem pos, komunikasi efektif, manajemen inve...",NONE,lebih dari 3 tahun di industri kopi. terampil ...,SMA,lorem,sma negeri 3 jakarta agustus 2017 - juli 2020 ...,Summary: Experience: lebih dari 3 tahun di ind...
4,John Doe,john.doe@email.com,62898765432100,assisted in planning and executing marketing c...,a recent high school graduate with a strong ac...,internship - marketing assistant august 2022 -...,D3,lorem,high school diploma august 2020 - july 2023 lo...,Summary: a recent high school graduate with a ...


In [39]:
print(len(all_results))

366


In [40]:
from deep_translator import GoogleTranslator
from tqdm import tqdm
import pandas as pd
import re

# progress bar
tqdm.pandas()

# ================= TRANSLATE =================
def safe_translate(text):

    if pd.isna(text):
        return ""

    try:
        return GoogleTranslator(
            source='auto',
            target='en'
        ).translate(str(text))

    except Exception as e:
        print("Translation error:", e)
        return str(text)

# translate kolom text
cv_compress["translated_text"] = cv_compress["text"].progress_apply(safe_translate)

# ================= CLEAN TEXT =================
def clean_text(text):

    text = str(text).lower()

    # hapus simbol
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

cv_compress["clean_text"] = cv_compress["translated_text"].apply(clean_text)

# ================= SAVE =================
cv_compress.to_csv("parsed_cv_final.csv", index=False)

print("✅ File berhasil disimpan: parsed_cv_final.csv")

# preview
cv_compress.head()

100%|██████████| 366/366 [00:24<00:00, 14.70it/s]


✅ File berhasil disimpan: parsed_cv_final.csv


,candidate_name,email,phone,skills,summary,experience,degree,university,education,text,translated_text,clean_text
0,Siti Nurhaliza,siti.nurhaliza@example.com,6281234567890,"penanganan keluhan, komunikasi efektif, penyel...",NONE,lebih dari 2 tahun dalam menangani pertanyaan ...,SMA,lorem,sma negeri 2 jakarta agustus 2018 - juli 2021 ...,Summary: Experience: lebih dari 2 tahun dalam ...,Summary: Experience: more than 2 years in effi...,summary experience more than 2 years in effici...
1,Rina Aulia,rina.aulia@example.com,6281234567890,"komunikasi efektif, crm software, salesforce, ...",NONE,3 tahun dalam memberikan pelayanan yang optima...,D3,universitas lorem ipsum jakarta lulus dengan p...,d3 administrasi bisnis september 2018 - juli 2...,Summary: Experience: 3 tahun dalam memberikan ...,Summary: Experience: 3 years in providing opti...,summary experience 3 years in providing optima...
2,Anna Suryani,anna.suryani@example.com,6281234567890,"problem-solving, intern midwife august 2023 - ...",NONE,d in providing comprehensive care to expectant...,S1,university of indonesia jakarta focused on mat...,", and teamwork within a healthcare environment...",Summary: Experience: d in providing comprehens...,Summary: Experience: d in providing comprehens...,summary experience d in providing comprehensiv...
3,Andi Pratama,andi.pratama@example.com,6281123456789,"sistem pos, komunikasi efektif, manajemen inve...",NONE,lebih dari 3 tahun di industri kopi. terampil ...,SMA,lorem,sma negeri 3 jakarta agustus 2017 - juli 2020 ...,Summary: Experience: lebih dari 3 tahun di ind...,Summary: Experience: more than 3 years in the ...,summary experience more than 3 years in the co...
4,John Doe,john.doe@email.com,62898765432100,assisted in planning and executing marketing c...,a recent high school graduate with a strong ac...,internship - marketing assistant august 2022 -...,D3,lorem,high school diploma august 2020 - july 2023 lo...,Summary: a recent high school graduate with a ...,Summary: a recent high school graduate with a ...,summary a recent high school graduate with a s...


In [41]:
cv_compress = pd.read_csv("/content/parsed_cv_final.csv")
cv_compress.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,education,text,translated_text,clean_text
0,Siti Nurhaliza,siti.nurhaliza@example.com,6281234567890,"penanganan keluhan, komunikasi efektif, penyel...",NONE,lebih dari 2 tahun dalam menangani pertanyaan ...,SMA,lorem,sma negeri 2 jakarta agustus 2018 - juli 2021 ...,Summary: Experience: lebih dari 2 tahun dalam ...,Summary: Experience: more than 2 years in effi...,summary experience more than 2 years in effici...
1,Rina Aulia,rina.aulia@example.com,6281234567890,"komunikasi efektif, crm software, salesforce, ...",NONE,3 tahun dalam memberikan pelayanan yang optima...,D3,universitas lorem ipsum jakarta lulus dengan p...,d3 administrasi bisnis september 2018 - juli 2...,Summary: Experience: 3 tahun dalam memberikan ...,Summary: Experience: 3 years in providing opti...,summary experience 3 years in providing optima...
2,Anna Suryani,anna.suryani@example.com,6281234567890,"problem-solving, intern midwife august 2023 - ...",NONE,d in providing comprehensive care to expectant...,S1,university of indonesia jakarta focused on mat...,", and teamwork within a healthcare environment...",Summary: Experience: d in providing comprehens...,Summary: Experience: d in providing comprehens...,summary experience d in providing comprehensiv...
3,Andi Pratama,andi.pratama@example.com,6281123456789,"sistem pos, komunikasi efektif, manajemen inve...",NONE,lebih dari 3 tahun di industri kopi. terampil ...,SMA,lorem,sma negeri 3 jakarta agustus 2017 - juli 2020 ...,Summary: Experience: lebih dari 3 tahun di ind...,Summary: Experience: more than 3 years in the ...,summary experience more than 3 years in the co...
4,John Doe,john.doe@email.com,62898765432100,assisted in planning and executing marketing c...,a recent high school graduate with a strong ac...,internship - marketing assistant august 2022 -...,D3,lorem,high school diploma august 2020 - july 2023 lo...,Summary: a recent high school graduate with a ...,Summary: a recent high school graduate with a ...,summary a recent high school graduate with a s...
